# Penerapan Metode IndoBERT untuk Mengukur Kemiripan Berdasarkan Judul Video pada Sistem Rekomendasi Channel YouTube Indonesia

**Metodologi:** Content-Based Filtering menggunakan IndoBERT + Cosine Similarity

**Alur Pipeline:**
1. Install & Import Library
2. Load Dataset JSON
3. Text Cleaning
4. Case Folding
5. Tokenisasi dengan Stanza (Bahasa Indonesia)
6. Split Data untuk Evaluasi (80/20 per channel: Candidate Set & Query Set)
7. Load Model IndoBERT (Pre-trained)
8. Generate Embedding Judul Video (Masked Mean Pooling)
9. Agregasi Vektor per Channel (Mean Pooling + L2 Normalization)
10. Cosine Similarity Matrix (Mengukur Kemiripan Antar Channel)
11. Fungsi Rekomendasi (Top-K dari Candidate Set)
12. Evaluasi Sistem (Precision@K pada Query Set)
13. Simpan Artefak (Opsional)

---
## 1. Install & Import Library

In [1]:
# Install library yang diperlukan
%pip install torch transformers stanza scikit-learn pandas numpy tqdm --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
import os
import numpy as np
import pandas as pd

# Deep Learning & NLP
import torch
from transformers import AutoTokenizer, AutoModel

# Stanza untuk Tokenisasi Bahasa Indonesia
import stanza

# Similarity & Evaluation
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

# Progress bar
from tqdm import tqdm

# Konfigurasi device (GPU jika tersedia, fallback ke CPU jika tidak kompatibel)
def get_device():
    """Deteksi device dengan fallback ke CPU jika CUDA tidak kompatibel."""
    if not torch.cuda.is_available():
        return torch.device('cpu')

    try:
        major, minor = torch.cuda.get_device_capability(0)
        # PyTorch build yang terpasang di environment ini hanya mendukung CC >= 7.5
        if (major, minor) < (7, 5):
            print(
                f"⚠️  GPU {torch.cuda.get_device_name(0)} memiliki compute capability "
                f"sm_{major}{minor}, jadi fallback ke CPU"
            )
            return torch.device('cpu')

        # Test ringan agar benar-benar memastikan CUDA bisa dipakai
        _ = torch.randn(1, device='cuda')
        return torch.device('cuda')
    except Exception as e:
        print(f"⚠️  CUDA tidak bisa dipakai ({e}) → fallback ke CPU")
        return torch.device('cpu')

device = get_device()
print(f'Menggunakan device  : {device}')
print(f'PyTorch version     : {torch.__version__}')
print('Import library selesai.')

/home/candimadam/Documents/Final Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⚠️  GPU NVIDIA GeForce MX350 memiliki compute capability sm_61, jadi fallback ke CPU
Menggunakan device  : cpu
PyTorch version     : 2.11.0+cu130
Import library selesai.


/home/candimadam/Documents/Final Project/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:371: UserWarning: Found GPU0 NVIDIA GeForce MX350 which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/candimadam/Documents/Final Project/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:489: UserWarning: 
NVIDIA GeForce MX350 with CUDA capability sm_61 is not compatible with the c

---
## 2. Load Dataset JSON

In [3]:
# Path file dataset
DATA_PATH = 'data_video.json'

# Load JSON
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# Konversi ke DataFrame
df = pd.DataFrame(raw_data)

# Informasi Dataset
print('=' * 55)
print('           INFORMASI DATASET')
print('=' * 55)
print(f'Total record (video)   : {len(df):,}')
print(f'Total kolom            : {len(df.columns)}')
print(f'Nama kolom             : {list(df.columns)}')
print(f'Jumlah channel unik    : {df["nama_channel"].nunique()}')
print(f'Jumlah kategori unik   : {df["kategori"].nunique()}')
print('=' * 55)

# Distribusi video per channel
dist = df['nama_channel'].value_counts()
print(f'\nDistribusi jumlah video per channel:')
print(f'  Min  : {dist.min()} video')
print(f'  Max  : {dist.max()} video')
print(f'  Rata : {dist.mean():.1f} video')
print()

# Tampilkan 5 baris pertama
df.head()

           INFORMASI DATASET
Total record (video)   : 10,000
Total kolom            : 9
Nama kolom             : ['id', 'link_channel', 'nama_channel', 'kategori', 'jumlah_pelanggan', 'judul', 'link', 'jumlah_tayangan', 'tanggal_upload']
Jumlah channel unik    : 100
Jumlah kategori unik   : 10

Distribusi jumlah video per channel:
  Min  : 100 video
  Max  : 100 video
  Rata : 100.0 video



,id,link_channel,nama_channel,kategori,jumlah_pelanggan,judul,link,jumlah_tayangan,tanggal_upload
0,1,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP...,https://www.youtube.com/watch?v=zE5H9KQ_Hyg,1700000,5 days ago
1,2,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,RAJA TERAKHIR HP SAMSUNG!,https://www.youtube.com/watch?v=snB4jbtscxU,932000,10 days ago
2,3,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...,https://www.youtube.com/watch?v=3KBJtbEAdzs,802000,11 days ago
3,4,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,"Kalau Apple niat, iPhone bisa seworth it ini.....",https://www.youtube.com/watch?v=rkLpVyRGCPw,1300000,2 weeks ago
4,5,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,Xiaomi pun ngeluh soal fenomena ini...,https://www.youtube.com/watch?v=Z4m_fwJ5eHQ&pp...,1200000,2 weeks ago


In [4]:
# Daftar channel yang tersedia 
print('Daftar Channel YouTube dalam Dataset:')
print('-' * 50)
for i, (ch, cnt) in enumerate(df['nama_channel'].value_counts().items(), 1):
    print(f'{i:>3}. {ch:<40} ({cnt} video)')

Daftar Channel YouTube dalam Dataset:
--------------------------------------------------
  1. @GadgetIn                                (100 video)
  2. @JagatReview                             (100 video)
  3. @GadgetGaul                              (100 video)
  4. @DHIARCOM                                (100 video)
  5. @DKIDchannel                             (100 video)
  6. @PricebookIndonesia                      (100 video)
  7. @Sobat_HAPE                              (100 video)
  8. @projectreview                           (100 video)
  9. @K2G                                     (100 video)
 10. @YoutuberCupu                            (100 video)
 11. @NexCarlos                               (100 video)
 12. @riasukmawijaya                          (100 video)
 13. @tanboykun                               (100 video)
 14. @KUBILER                                 (100 video)
 15. @MamankKuliner                           (100 video)
 16. @Melkibajaj                         

---
## 3. Text Cleaning

Pembersihan teks secara **minimalis**: hanya menghapus noise tanpa makna bahasa
(emoji, simbol dekoratif non-standar, karakter encoding rusak).
Tanda baca standar **(titik, koma, tanda tanya)** tetap dipertahankan karena IndoBERT memanfaatkannya.

    Tahapan:
    1. Menghapus emoji dan simbol Unicode non-standar
    2. Menghapus karakter encoding yang rusak / tidak dikenali
    3. Menghapus karakter khusus / dekoratif yang bukan tanda baca standar
    4. Merapikan spasi berlebih

In [5]:
def text_cleaning(text):
    if not isinstance(text, str):
        return ''

    # 1. Hapus emoji dan simbol Unicode non-standar
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"   # emoticons wajah
        "\U0001F300-\U0001F5FF"   # simbol & piktogram
        "\U0001F680-\U0001F6FF"   # transport & peta
        "\U0001F1E0-\U0001F1FF"   # bendera
        "\U00002600-\U000026FF"   # simbol umum
        "\U00002700-\U000027BF"   # Dingbats
        "\U0001F900-\U0001F9FF"   # simbol tambahan
        "\U0001FA00-\U0001FA6F"   # simbol tambahan-A
        "\U0001FA70-\U0001FAFF"   # simbol tambahan-B
        "\U00002300-\U000023FF"   # teknis
        "]+",
        flags=re.UNICODE
    )
    text = emoji_pattern.sub(' ', text)

    # 2. Hapus karakter non-printable / encoding rusak
    text = re.sub(r'[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F]', ' ', text)

    # 3. Hapus simbol dekoratif yang bukan tanda baca standar
    #    Pertahankan: huruf, angka, spasi, . , ? ! - ( ) / @ # % + = : ; ' "
    text = re.sub(r'[^\w\s.,?!\-()/@#%+=\'":;]', ' ', text, flags=re.UNICODE)

    # 4. Rapikan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Terapkan Text Cleaning
df['judul_clean'] = df['judul'].apply(text_cleaning)

# Tampilkan contoh hasil cleaning
print('Contoh Hasil Text Cleaning:')
print('=' * 70)
for _, row in df[['judul', 'judul_clean']].head(8).iterrows():
    print(f'SEBELUM : {row["judul"]}')
    print(f'SESUDAH : {row["judul_clean"]}')
    print('-' * 70)

Contoh Hasil Text Cleaning:
SEBELUM : Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP ASLINYA...
SESUDAH : Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP ASLINYA...
----------------------------------------------------------------------
SEBELUM : RAJA TERAKHIR HP SAMSUNG!
SESUDAH : RAJA TERAKHIR HP SAMSUNG!
----------------------------------------------------------------------
SEBELUM : Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...
SESUDAH : Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...
----------------------------------------------------------------------
SEBELUM : Kalau Apple niat, iPhone bisa seworth it ini... - Review iPhone 17
SESUDAH : Kalau Apple niat, iPhone bisa seworth it ini... - Review iPhone 17
----------------------------------------------------------------------
SEBELUM : Xiaomi pun ngeluh soal fenomena ini...
SESUDAH : Xiaomi pun ngeluh soal fenomena ini...
----------------------------------------------------------------------
SEBELUM : Rekomendasi HP TERBAIK buat A

---
## 4. Case Folding

Mengubah seluruh teks menjadi huruf kecil (**lowercase**) agar sesuai dengan
model **IndoBERT Base Uncased** yang dilatih pada teks huruf kecil.

In [6]:
def case_folding(text):
    if not isinstance(text, str):
        return ''
    return text.lower()

# Terapkan case folding setelah text cleaning
df['judul_lower'] = df['judul_clean'].apply(case_folding)

# Tampilkan contoh
print('Contoh Hasil Case Folding (setelah Text Cleaning):')
print('=' * 70)
for _, row in df[['judul_clean', 'judul_lower']].head(5).iterrows():
    print(f'SEBELUM : {row["judul_clean"]}')
    print(f'SESUDAH : {row["judul_lower"]}')
    print('-' * 70)

Contoh Hasil Case Folding (setelah Text Cleaning):
SEBELUM : Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP ASLINYA...
SESUDAH : unboxing iphone 17 pro palsu yang sangat mirip aslinya...
----------------------------------------------------------------------
SEBELUM : RAJA TERAKHIR HP SAMSUNG!
SESUDAH : raja terakhir hp samsung!
----------------------------------------------------------------------
SEBELUM : Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...
SESUDAH : rp1.599 juta! ketika oppo niat bikin hp murah...
----------------------------------------------------------------------
SEBELUM : Kalau Apple niat, iPhone bisa seworth it ini... - Review iPhone 17
SESUDAH : kalau apple niat, iphone bisa seworth it ini... - review iphone 17
----------------------------------------------------------------------
SEBELUM : Xiaomi pun ngeluh soal fenomena ini...
SESUDAH : xiaomi pun ngeluh soal fenomena ini...
----------------------------------------------------------------------


---
## 5. Tokenisasi dengan Stanza (Bahasa Indonesia)

Stanza digunakan untuk **tokenisasi** teks Bahasa Indonesia.
Output berupa teks yang sudah dinormalisasi (token yang bergabung kembali).

In [7]:
# Langkah 1: Download model Bahasa Indonesia
stanza.download('id', verbose=False)   # 'id' = kode bahasa Indonesia
print('Model Stanza Bahasa Indonesia siap.')

Model Stanza Bahasa Indonesia siap.


In [8]:
# Langkah 2: Inisialisasi pipeline Stanza 
# Hanya processor 'tokenize' yang diaktifkan untuk efisiensi

try:
    nlp_stanza = stanza.Pipeline(
        lang='id',
        processors='tokenize',
        verbose=False
    )
    print("✓ Stanza initialized on GPU")
except RuntimeError as e:
    print(f"⚠️  GPU error: {str(e)[:80]}...")
    print("   Fallback ke CPU...")
    nlp_stanza = stanza.Pipeline(
        lang='id',
        processors='tokenize',
        device='cpu',
        verbose=False
    )
    print("✓ Stanza initialized on CPU")

def tokenize_stanza(text):
    if not text or not text.strip():
        return text

    doc = nlp_stanza(text)
    # Gabungkan semua token dari semua kalimat
    tokens = []
    for sentence in doc.sentences:
        for token in sentence.tokens:
            tokens.append(token.text)

    return ' '.join(tokens)

# Langkah 3: Terapkan tokenisasi ke seluruh dataset
print(f'Memproses tokenisasi Stanza untuk {len(df):,} judul...')
print('(Proses ini memerlukan beberapa menit)\n')

tqdm.pandas(desc='Tokenisasi Stanza')
df['judul_tokenized'] = df['judul_lower'].progress_apply(tokenize_stanza)

print('\nTokenisasi selesai!')
print('\nContoh hasil tokenisasi:')
print('=' * 70)
for _, row in df[['judul_lower', 'judul_tokenized']].head(5).iterrows():
    print(f'INPUT  : {row["judul_lower"]}')
    print(f'OUTPUT : {row["judul_tokenized"]}')
    print('-' * 70)

⚠️  GPU error: cuDNN version 91900 is not compatible with devices with SM < 7.5. Please install...
   Fallback ke CPU...
✓ Stanza initialized on CPU
Memproses tokenisasi Stanza untuk 10,000 judul...
(Proses ini memerlukan beberapa menit)



Tokenisasi Stanza: 100%|██████████| 10000/10000 [02:13<00:00, 75.15it/s]


Tokenisasi selesai!

Contoh hasil tokenisasi:
INPUT  : unboxing iphone 17 pro palsu yang sangat mirip aslinya...
OUTPUT : unboxing iphone 17 pro palsu yang sangat mirip aslinya . . .
----------------------------------------------------------------------
INPUT  : raja terakhir hp samsung!
OUTPUT : raja terakhir hp samsung !
----------------------------------------------------------------------
INPUT  : rp1.599 juta! ketika oppo niat bikin hp murah...
OUTPUT : rp1.599 juta ! ketika oppo niat bikin hp murah . . .
----------------------------------------------------------------------
INPUT  : kalau apple niat, iphone bisa seworth it ini... - review iphone 17
OUTPUT : kalau apple niat , iphone bisa seworth it ini . . . - review iphone 17
----------------------------------------------------------------------
INPUT  : xiaomi pun ngeluh soal fenomena ini...
OUTPUT : xiaomi pun ngeluh soal fenomena ini . . .
----------------------------------------------------------------------


---
## 6. Split Data untuk Evaluasi (80/20 per channel — Candidate & Query Sets)

Pembagian data ini digunakan untuk memisahkan:
- **Candidate Set (80%)** = Katalog channel yang tersedia untuk direkomendasikan
- **Query Set / Evaluation Set (20%)** = Channel input untuk testing sistem rekomendasi


In [9]:
from sklearn.model_selection import train_test_split

channel_level_df = (
    df.drop_duplicates(subset='nama_channel')[['nama_channel', 'kategori']]
      .reset_index(drop=True)
)

all_channels = channel_level_df['nama_channel'].tolist()
all_categories = channel_level_df['kategori'].tolist()

# Split: 80% = Candidate Set (katalog rekomendasi), 20% = Query Set (testing)
candidate_channels, query_channels = train_test_split(
    all_channels,
    test_size=0.2,
    random_state=42,
    stratify=all_categories
)

candidate_df = df[df['nama_channel'].isin(candidate_channels)].copy()
query_df     = df[df['nama_channel'].isin(query_channels)].copy()

print('=' * 75)
print('          HASIL SPLIT DATA - UNSUPERVISED EVALUATION SETUP')
print('=' * 75)
print(f'Total channel                : {len(all_channels):,}')
print(f'Candidate Set (katalog)      : {len(candidate_channels):,} channel')
print(f'Query Set (evaluasi)         : {len(query_channels):,} channel')
print('-' * 75)
print(f'Videos di Candidate Set      : {len(candidate_df):,}')
print(f'Videos di Query Set          : {len(query_df):,}')
print('-' * 75)
print('Distribusi kategori per set:')
print('• Candidate :', candidate_df['kategori'].value_counts().to_dict())
print('• Query     :', query_df['kategori'].value_counts().to_dict())
print('=' * 75)

          HASIL SPLIT DATA - UNSUPERVISED EVALUATION SETUP
Total channel                : 100
Candidate Set (katalog)      : 80 channel
Query Set (evaluasi)         : 20 channel
---------------------------------------------------------------------------
Videos di Candidate Set      : 8,000
Videos di Query Set          : 2,000
---------------------------------------------------------------------------
Distribusi kategori per set:
• Candidate : {'Gadgets': 800, 'Food': 800, 'Gaming': 800, 'Entertainment': 800, 'Education': 800, 'Automotive': 800, 'Sports': 800, 'Music': 800, 'News': 800, 'Animals': 800}
• Query     : {'Gadgets': 200, 'Food': 200, 'Gaming': 200, 'Entertainment': 200, 'Education': 200, 'Automotive': 200, 'Sports': 200, 'Music': 200, 'News': 200, 'Animals': 200}


---
## 7. Load Model IndoBERT (Pre-trained)

Menggunakan model **`indolem/indobert-base-uncased`** dari Hugging Face.
Model ini dilatih khusus untuk Bahasa Indonesia.

In [10]:
INDOBERT_MODEL_NAME = 'indolem/indobert-base-uncased'

print(f'Memuat model IndoBERT: {INDOBERT_MODEL_NAME}')
print('(Download pertama kali memerlukan beberapa menit)\n')

# Load Tokenizer IndoBERT 
tokenizer = AutoTokenizer.from_pretrained(INDOBERT_MODEL_NAME)
print(f'Tokenizer berhasil dimuat.')
print(f'  Vocab size  : {tokenizer.vocab_size:,}')

# Load Model IndoBERT
model = AutoModel.from_pretrained(INDOBERT_MODEL_NAME)
model = model.to(device)   # pindahkan ke GPU jika tersedia
model.eval()               # mode evaluasi (non-training)

print(f'\nModel berhasil dimuat → device: {device}')
print(f'  Hidden size : {model.config.hidden_size}')
print(f'  Num layers  : {model.config.num_hidden_layers}')
print(f'  Num heads   : {model.config.num_attention_heads}')

Memuat model IndoBERT: indolem/indobert-base-uncased
(Download pertama kali memerlukan beberapa menit)



Tokenizer berhasil dimuat.
  Vocab size  : 31,923


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11679.70it/s]
BertModel LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model berhasil dimuat → device: cpu
  Hidden size : 768
  Num layers  : 12
  Num heads   : 12


---
## 8. Generate Embedding Judul Video (Masked Mean Pooling)

Setiap judul video diubah menjadi **embedding vector** menggunakan
**Masked Mean Pooling** dari output last hidden state IndoBERT.

Masked Mean Pooling menghitung rata-rata embedding token dengan mempertimbangkan
attention mask dari tokenizer, sehingga padding token tidak mempengaruhi hasil.

Catatan:
- Di **Langkah 8**, mean dilakukan pada level token (masked mean pooling, padding diabaikan)

    Parameters
    - texts      : list[str] – daftar judul yang sudah dipreproses
    - tokenizer  : IndoBERT tokenizer
    - model      : IndoBERT model
    - device     : torch.device – CPU atau GPU
    - batch_size : int – jumlah teks per batch
    - max_length : int – panjang token maksimum (default 128)

In [11]:
def generate_all_embeddings(texts, tokenizer, model, device, batch_size=32, max_length=128):
    # Menampung embedding dari semua batch
    all_embeddings = []

    for start in tqdm(range(0, len(texts), batch_size), desc='Generating embeddings', unit='batch'):
        batch_texts = texts[start:start + batch_size]

        # Tokenisasi batch
        inputs = tokenizer(
            batch_texts,
            return_tensors='pt',
            max_length=max_length,
            truncation=True,
            padding=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Forward pass tanpa gradient
        with torch.no_grad():
            token_embeddings = model(**inputs).last_hidden_state  # (B, L, 768)

        # Masked Mean Pooling: rata-rata embedding dengan attention mask
        mask = inputs['attention_mask'].unsqueeze(-1).float()
        summed = (token_embeddings * mask).sum(dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        mean_embeddings = summed / counts

        all_embeddings.append(mean_embeddings.cpu().numpy())

    # Gabungkan semua batch -> (n_texts, 768)
    return np.vstack(all_embeddings)

# Jalankan proses embedding
print(f'Memulai embedding untuk {len(df):,} judul video...')
print('Metode: Masked Mean Pooling')
print(f'Perangkat eksekusi: {device}')
print('(Gunakan GPU hanya jika kompatibel; jika tidak, otomatis CPU)\n')

texts_to_embed = df['judul_tokenized'].tolist()
video_embeddings = generate_all_embeddings(
    texts=texts_to_embed,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=32
)

# Simpan embedding ke DataFrame
df['embedding'] = list(video_embeddings)

print(f'\nEmbedding berhasil dibuat!')
print(f'  Shape matrix  : {video_embeddings.shape}')
print(f'  Jumlah video  : {video_embeddings.shape[0]:,}')
print(f'  Dimensi vektor: {video_embeddings.shape[1]}')
print(f'  Dtype         : {video_embeddings.dtype}')
print(f'  Metode        : Masked Mean Pooling')

Memulai embedding untuk 10,000 judul video...
Metode: Masked Mean Pooling
Perangkat eksekusi: cpu
(Gunakan GPU hanya jika kompatibel; jika tidak, otomatis CPU)



Generating embeddings:   0%|          | 0/313 [00:00<?, ?batch/s]Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/home/candimadam/Documents/Final Project/.venv/lib/python3.12/site-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/home/candimadam/Documents/Final Project/.venv/lib/python3.12/site-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/candimadam/Documents/Final Project/.venv/lib/python3.12/site-packages/transformers/safetensors_conversion.py", line 72, in get_conversi


Embedding berhasil dibuat!
  Shape matrix  : (10000, 768)
  Jumlah video  : 10,000
  Dimensi vektor: 768
  Dtype         : float32
  Metode        : Masked Mean Pooling


---
## 9. Agregasi Vektor Channel (Mean Pooling + L2 Normalization)

Setiap channel memiliki banyak video, sehingga seluruh embedding video dirata-ratakan
(**mean pooling**) untuk menghasilkan **satu vektor representasi per channel**.

Catatan:
- Di **Langkah 9**, mean dilakukan pada level video per channel (tanpa token padding)

Setelah agregasi, diterapkan **L2 Normalization** agar perhitungan cosine similarity lebih stabil.

```
Channel A -> [emb_vid1, emb_vid2, ..., emb_vidN]
          -> mean(emb_vid1 ... emb_vidN)
          -> 1 vektor channel (768-dim)
```

    Parameters
    - df : DataFrame dengan kolom `nama_channel` dan `embedding`

In [12]:
from sklearn.preprocessing import normalize

def aggregate_channel_embeddings(df):
    # Hitung rata-rata embedding per channel
    grouped = df.groupby('nama_channel')
    return {
        ch: np.mean(np.stack(group['embedding'].values), axis=0)
        for ch, group in tqdm(grouped, desc='Agregasi channel', unit='channel')
    }

# Jalankan agregasi
channel_vectors = aggregate_channel_embeddings(df)
channel_names = list(channel_vectors.keys())
channel_matrix = np.stack([channel_vectors[ch] for ch in channel_names])

# L2 normalization agar cosine lebih stabil
channel_matrix = normalize(channel_matrix, norm='l2')

# Sinkronkan kembali ke dictionary
for i, ch in enumerate(channel_names):
    channel_vectors[ch] = channel_matrix[i]

print(f'\nAgregasi selesai!')
print(f'  Jumlah channel  : {len(channel_names)}')
print(f'  Shape matrix    : {channel_matrix.shape}')
print(f'  L2 Normalization: ✓ (semua vektor ber-norm = 1.0)')

# Verifikasi norm
norms = np.linalg.norm(channel_matrix, axis=1)
print(f'  Norm (min-max)  : {norms.min():.4f} – {norms.max():.4f}  (seharusnya ~1.0)')

# Tampilkan contoh 5 channel
print('\nContoh vektor channel (5 nilai pertama dari vektor):')
print('=' * 65)
for ch in channel_names[:5]:
    vec = channel_vectors[ch]
    n_video = len(df[df['nama_channel'] == ch])
    print(f'{ch:<40} | {n_video:>3} video | vec[:5]={vec[:5].round(4)}')


Agregasi channel: 100%|██████████| 100/100 [00:00<00:00, 1776.24channel/s]


Agregasi selesai!
  Jumlah channel  : 100
  Shape matrix    : (100, 768)
  L2 Normalization: ✓ (semua vektor ber-norm = 1.0)
  Norm (min-max)  : 1.0000 – 1.0000  (seharusnya ~1.0)

Contoh vektor channel (5 nilai pertama dari vektor):
@AfifYulistian                           | 100 video | vec[:5]=[-0.0079 -0.019  -0.0769  0.0283 -0.0062]
@AlshadAhmad                             | 100 video | vec[:5]=[-0.0099 -0.0274 -0.0733  0.0075 -0.032 ]
@Anak.Kuliner                            | 100 video | vec[:5]=[-0.0083 -0.0298 -0.073   0.0217 -0.0301]
@AquariusMusikindo                       | 100 video | vec[:5]=[-0.0083 -0.0175 -0.0699  0.0178 -0.0201]
@ArisSportTv                             | 100 video | vec[:5]=[-0.0151 -0.0172 -0.076   0.014  -0.0357]


---
## 10. Cosine Similarity Matrix (Mengukur Kemiripan Antar Channel)

Menghitung kemiripan antar channel menggunakan **Cosine Similarity**.

Awalnya berupa matriks dengan setiap baris adalah vektor representasi channel **(n_channels × 768)**.

Hasilnya berupa matriks simetris berukuran **(n_channel × n_channel)**.

In [13]:
similarity_matrix = cosine_similarity(channel_matrix)

# Konversi ke DataFrame agar mudah diakses via nama channel
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=channel_names,
    columns=channel_names
)

print('Cosine Similarity Matrix berhasil dibuat!')
print(f'  Shape : {similarity_matrix.shape}')
print(f'  (Baris & Kolom = {len(channel_names)} channel)')

# Statistik nilai similarity non-diagonal (antar channel berbeda)
mask     = ~np.eye(len(channel_names), dtype=bool)
off_diag = similarity_matrix[mask]

print(f'\nStatistik Cosine Similarity (antar channel berbeda):')
print(f'  Min  : {off_diag.min():.4f}')
print(f'  Max  : {off_diag.max():.4f}')
print(f'  Mean : {off_diag.mean():.4f}')
print(f'  Std  : {off_diag.std():.4f}')

# Tampilkan sebagian matriks (5×5)
print('\nCuplikan Similarity Matrix (5×5):')
similarity_df.iloc[:5, :5].round(4)

Cosine Similarity Matrix berhasil dibuat!
  Shape : (100, 100)
  (Baris & Kolom = 100 channel)

Statistik Cosine Similarity (antar channel berbeda):
  Min  : 0.6354
  Max  : 0.9884
  Mean : 0.9022
  Std  : 0.0436

Cuplikan Similarity Matrix (5×5):


,@AfifYulistian,@AlshadAhmad,@Anak.Kuliner,@AquariusMusikindo,@ArisSportTv
@AfifYulistian,1.0000,0.9358,0.9367,0.9110,0.9123
@AlshadAhmad,0.9358,1.0000,0.9404,0.8776,0.9246
@Anak.Kuliner,0.9367,0.9404,1.0000,0.8590,0.8848
@AquariusMusikindo,0.9110,0.8776,0.8590,1.0000,0.9097
@ArisSportTv,0.9123,0.9246,0.8848,0.9097,1.0000


---
## 11. Fungsi Rekomendasi (Menampilkan Top-K Channel Paling Mirip)

Fungsi `recommend_channel(...)` mengembalikan **Top-K channel paling mirip**
berdasarkan Cosine Similarity dari vektor representasi channel.

    Alur:
    1. Ambil baris similarity channel input dari similarity_df.
    2. Hilangkan channel itu sendiri.
    3. Batasi kandidat ke Candidate Set (jika candidate_channels diisi).
    4. Urutkan berdasarkan similarity tertinggi.
    5. Kembalikan Top-K channel beserta info pendukung.

    Parameters:
    - channel_name : str – nama channel input
    - top_k : int – jumlah rekomendasi yang dikembalikan
    - similarity_df : DataFrame – matriks similarity (index & columns = nama channel)
    - channel_info : DataFrame – info channel dengan index=nama_channel, columns=['kategori', 'jumlah_pelanggan']
    - candidate_channels : list[str] – daftar channel kandidat yang diizinkan

In [14]:
def recommend_channel(channel_name, top_k=5, similarity_df=None, channel_info=None, candidate_channels=None):
    # Validasi input
    if similarity_df is None or channel_info is None:
        raise ValueError("similarity_df dan channel_info harus diberikan sebagai parameter")

    if channel_name not in similarity_df.index:
        available = list(similarity_df.index)
        raise ValueError(
            f"Channel '{channel_name}' tidak ditemukan.\n"
            f"Channel tersedia: {available}"
        )

    # Ambil skor similarity lalu buang channel dirinya sendiri
    sim_scores = similarity_df.loc[channel_name].drop(labels=channel_name)

    # Jika kandidat dibatasi, ambil hanya channel dari Candidate Set
    if candidate_channels is not None:
        valid_candidates = [ch for ch in candidate_channels if ch in sim_scores.index]
        sim_scores = sim_scores.loc[valid_candidates]

    # Urutkan skor tertinggi dan ambil Top-K
    top_k_channels = sim_scores.sort_values(ascending=False).head(top_k)

    results = []
    for rank, (ch, score) in enumerate(top_k_channels.items(), start=1):
        if ch in channel_info.index:
            info = channel_info.loc[ch]
            kategori = info['kategori']
            pelanggan = info['jumlah_pelanggan']
        else:
            kategori = '-'
            pelanggan = 0

        results.append({
            'rank': rank,
            'nama_channel': ch,
            'kategori': kategori,
            'jumlah_pelanggan': pelanggan,
            'similarity_score': round(float(score), 4)
        })

    return pd.DataFrame(results).set_index('rank')

# ─── Siapkan data untuk fungsi ────────────────────────────────
channel_info_for_recommendation = (
    df.drop_duplicates(subset='nama_channel')
      .set_index('nama_channel')[['kategori', 'jumlah_pelanggan']]
)

# ─── Contoh Penggunaan ────────────────────────────────────────
print('=' * 70)
print('           CONTOH REKOMENDASI CHANNEL (dari Candidate Set)')
print('=' * 70)

input_channel = channel_names[0]  # channel pertama dalam dataset
TOP_K = 5

print(f'Channel Input : {input_channel}')
print(f'Top-K         : {TOP_K}')
print(f'Sumber        : Candidate Set (80% data)')
print('-' * 70)

rec_df = recommend_channel(
    channel_name=input_channel,
    top_k=TOP_K,
    similarity_df=similarity_df,
    channel_info=channel_info_for_recommendation,
    candidate_channels=candidate_channels  # Hanya rekomendasi dari Candidate Set
)

print(f'\nTop-{TOP_K} Channel yang Direkomendasikan untuk "{input_channel}":\n')
for idx, row in rec_df.iterrows():
    print(f'  {idx}. {row["nama_channel"]:<40} (similarity: {row["similarity_score"]:.4f})')

print()
print(rec_df.to_string())

           CONTOH REKOMENDASI CHANNEL (dari Candidate Set)
Channel Input : @AfifYulistian
Top-K         : 5
Sumber        : Candidate Set (80% data)
----------------------------------------------------------------------

Top-5 Channel yang Direkomendasikan untuk "@AfifYulistian":

  1. @DylandPROS                              (similarity: 0.9614)
  2. @WindahBasudara                          (similarity: 0.9578)
  3. @letdahyper                              (similarity: 0.9556)
  4. @ridwanhr                                (similarity: 0.9534)
  5. @nihongomantappu                         (similarity: 0.9523)

          nama_channel    kategori  jumlah_pelanggan  similarity_score
rank                                                                  
1          @DylandPROS      Gaming          16300000            0.9614
2      @WindahBasudara      Gaming          18200000            0.9578
3          @letdahyper      Gaming          10300000            0.9556
4            @ridwanhr  Aut

In [15]:
# ─── Uji Coba dengan 3 Channel Pertama (DEMO + label TRUE/FALSE kategori) ───
demo_channels = channel_names[:3]

# mapping channel -> kategori dari data yang sudah ada
channel_category = (
    df.drop_duplicates('nama_channel')
      .set_index('nama_channel')['kategori']
      .to_dict()
)

for ch in demo_channels:
    input_kategori = channel_category.get(ch, '-')

    print('=' * 80)
    print(f'Rekomendasi untuk: {ch} | Kategori input: {input_kategori}')
    print('=' * 80)

    rec = recommend_channel(
        channel_name=ch,
        top_k=5,
        similarity_df=similarity_df,
        channel_info=channel_info_for_recommendation,
        candidate_channels=candidate_channels  # Dari Candidate Set
    )

    for i, row in rec.iterrows():
        is_match = row['kategori'] == input_kategori
        status = 'TRUE (Relevan)' if is_match else 'FALSE (Beda kategori)'
        print(
            f'{i}. {row["nama_channel"]:<35} | '
            f'kategori: {row["kategori"]:<12} | '
            f'sim: {row["similarity_score"]:.4f} | '
            f'{status}'
        )
    print()

Rekomendasi untuk: @AfifYulistian | Kategori input: Gaming
1. @DylandPROS                         | kategori: Gaming       | sim: 0.9614 | TRUE (Relevan)
2. @WindahBasudara                     | kategori: Gaming       | sim: 0.9578 | TRUE (Relevan)
3. @letdahyper                         | kategori: Gaming       | sim: 0.9556 | TRUE (Relevan)
4. @ridwanhr                           | kategori: Automotive   | sim: 0.9534 | FALSE (Beda kategori)
5. @nihongomantappu                    | kategori: Education    | sim: 0.9523 | FALSE (Beda kategori)

Rekomendasi untuk: @AlshadAhmad | Kategori input: Animals
1. @farida.nurhan                      | kategori: Food         | sim: 0.9744 | FALSE (Beda kategori)
2. @BeemzAryo                          | kategori: Animals      | sim: 0.9697 | TRUE (Relevan)
3. @PANJIPETUALANG_REAL                | kategori: Animals      | sim: 0.9566 | TRUE (Relevan)
4. @letdahyper                         | kategori: Gaming       | sim: 0.9562 | FALSE (Beda kategori)

---
## 12. Evaluasi Sistem (Precision@K pada Query Set)

**Definisi relevansi:** channel dianggap relevan jika memiliki **kategori yang sama**
dengan channel input (ground truth berbasis kategori).

**Skema evaluasi:**
- **Query Set (20%)**: channel yang dijadikan input evaluasi
- **Candidate Set (80%)**: channel yang boleh muncul sebagai rekomendasi

    Parameters (precision@K untuk satu channel)
    - channel_name : str – channel yang dievaluasi
    - k : int – nilai K untuk Precision@K
    - similarity_df : DataFrame – matriks similarity
    - channel_category_map : dict – mapping channel ke kategori
    - candidate_channels : list[str] – daftar kandidat dari Candidate Set

    Parameters (evaluasi keseluruhan sistem)
    - k_values : list[int] – nilai K yang dievaluasi
    - eval_channels : list[str] – channel dari Query Set yang dievaluasi
    - similarity_df : DataFrame – matriks similarity
    - channel_category_map : dict – mapping channel ke kategori
    - candidate_channels : list[str] – daftar channel dari Candidate Set

In [16]:
def precision_at_k(channel_name, k, similarity_df=None, channel_category_map=None, candidate_channels=None):
    if similarity_df is None or channel_category_map is None:
        raise ValueError("similarity_df dan channel_category_map harus diberikan")

    if channel_name not in similarity_df.index:
        return 0.0

    target_category = channel_category_map.get(channel_name)
    if target_category is None:
        return 0.0

    # Ambil kandidat rekomendasi
    sim_scores = similarity_df.loc[channel_name].drop(labels=channel_name)
    if candidate_channels is not None:
        valid_candidates = [ch for ch in candidate_channels if ch in sim_scores.index]
        sim_scores = sim_scores.loc[valid_candidates]

    top_k_channels = sim_scores.sort_values(ascending=False).head(k).index.tolist()

    # Hitung relevan: kategori sama
    relevant_count = sum(channel_category_map.get(ch) == target_category for ch in top_k_channels)
    return relevant_count / k


def evaluate_system(k_values=None, eval_channels=None, similarity_df=None, 
                    channel_category_map=None, candidate_channels=None):
    if similarity_df is None or channel_category_map is None:
        raise ValueError("similarity_df dan channel_category_map harus diberikan")

    if k_values is None:
        k_values = [1, 3, 5, 10]
    if eval_channels is None:
        eval_channels = list(similarity_df.index)

    rows = []
    for ch in tqdm(eval_channels, desc='Evaluasi Precision@K', unit='channel'):
        row = {'channel': ch, 'kategori': channel_category_map.get(ch, '-')}
        for k in k_values:
            row[f'P@{k}'] = round(
                precision_at_k(ch, k, similarity_df, channel_category_map, candidate_channels),
                4
            )
        rows.append(row)

    eval_df = pd.DataFrame(rows).set_index('channel')

    # Tambahkan baris rata-rata
    avg_row = {'kategori': 'AVERAGE'}
    for k in k_values:
        avg_row[f'P@{k}'] = round(eval_df[f'P@{k}'].mean(), 4)

    avg_df = pd.DataFrame([avg_row], index=['--- AVERAGE ---'])
    return pd.concat([eval_df, avg_df])


# ─── Siapkan mapping kategori ─────────────────────────────────
channel_category_map = (
    df.drop_duplicates(subset='nama_channel')
      .set_index('nama_channel')['kategori']
      .to_dict()
)

# Jalankan evaluasi di Query Set
K_VALUES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

print('Menjalankan evaluasi Precision@K untuk Query Set...')
print('(Channel dari Query Set akan dicari rekomendasi dari Candidate Set)')
print()

eval_df = evaluate_system(
    k_values=K_VALUES,
    eval_channels=query_channels,  # Query Set (20% data)
    similarity_df=similarity_df,
    channel_category_map=channel_category_map,
    candidate_channels=candidate_channels  # Candidate Set (80% data)
)

print('\n' + '=' * 75)
print('    HASIL EVALUASI SISTEM REKOMENDASI (Precision@K - Query Set)')
print('=' * 75)
print(eval_df.to_string())
print('=' * 75)

# Ringkasan rata-rata
print('\nRingkasan Rata-Rata Precision@K untuk Query Set:')
avg_series = eval_df.loc['--- AVERAGE ---']
for k in K_VALUES:
    print(f'  Precision@{k:>2} = {avg_series[f"P@{k}"]:.4f}')

Menjalankan evaluasi Precision@K untuk Query Set...
(Channel dari Query Set akan dicari rekomendasi dari Candidate Set)



Evaluasi Precision@K: 100%|██████████| 20/20 [00:00<00:00, 88.41channel/s]


    HASIL EVALUASI SISTEM REKOMENDASI (Precision@K - Query Set)
                         kategori  P@1    P@2     P@3    P@4   P@5     P@6     P@7     P@8     P@9   P@10
@amarpdchannel            Animals  1.0  0.500  0.3333  0.500  0.60  0.5000  0.4286  0.3750  0.3333  0.300
@tvOneNews                   News  1.0  1.000  1.0000  1.000  1.00  0.8333  0.8571  0.8750  0.7778  0.700
@TirtaPengPengPeng      Education  0.0  0.000  0.0000  0.000  0.00  0.0000  0.0000  0.0000  0.0000  0.000
@motomobitv            Automotive  1.0  1.000  1.0000  0.750  0.60  0.6667  0.5714  0.5000  0.4444  0.400
@ybrap              Entertainment  0.0  0.000  0.0000  0.000  0.00  0.0000  0.0000  0.0000  0.1111  0.100
@IBLTV                     Sports  0.0  0.000  0.3333  0.250  0.20  0.3333  0.4286  0.3750  0.4444  0.400
@GarasiDrift           Automotive  1.0  0.500  0.6667  0.500  0.40  0.3333  0.2857  0.2500  0.2222  0.200
@AfifYulistian             Gaming  1.0  1.000  1.0000  0.750  0.60  0.5000  0.4286  0.5

In [17]:
# ─── Evaluasi Detail per Kategori ────────────────────────────
print('\nPrecision@K Per Kategori (rata-rata per kategori):')
print('=' * 55)

# Filter baris rata-rata
eval_df_clean = eval_df[eval_df['kategori'] != 'AVERAGE'].copy()

pk_cols = [f'P@{k}' for k in K_VALUES]

cat_summary = (
    eval_df_clean.groupby('kategori')[pk_cols]
    .mean()
    .round(4)
    .sort_values('P@5', ascending=False)
)

# Top-10 kategori terbaik berdasarkan kemiripan (pakai P@5)
top10_cat = cat_summary['P@5'].head(10)

# Simpan juga dalam bentuk list (ranked)
top10_cat_list = [
    {'rank': i, 'kategori': kategori, 'nilai_kemiripan': float(score)}
    for i, (kategori, score) in enumerate(top10_cat.items(), start=1)
]

print('\nTop 10 Kategori dengan Kemiripan Tertinggi (berdasarkan P@5):')
for item in top10_cat_list:
    print(f"{item['rank']:>2}. {item['kategori']:<20} -> P@5 = {item['nilai_kemiripan']:.4f}")
print()
print(cat_summary.to_string())


Precision@K Per Kategori (rata-rata per kategori):

Top 10 Kategori dengan Kemiripan Tertinggi (berdasarkan P@5):
 1. Gadgets              -> P@5 = 1.0000
 2. News                 -> P@5 = 0.9000
 3. Food                 -> P@5 = 0.7000
 4. Gaming               -> P@5 = 0.6000
 5. Automotive           -> P@5 = 0.5000
 6. Animals              -> P@5 = 0.5000
 7. Music                -> P@5 = 0.5000
 8. Entertainment        -> P@5 = 0.3000
 9. Education            -> P@5 = 0.2000
10. Sports               -> P@5 = 0.2000

               P@1   P@2     P@3    P@4  P@5     P@6     P@7     P@8     P@9  P@10
kategori                                                                          
Gadgets        1.0  1.00  1.0000  1.000  1.0  1.0000  0.8571  0.7500  0.6667  0.60
News           1.0  1.00  1.0000  1.000  0.9  0.7500  0.7857  0.8125  0.7778  0.70
Food           1.0  0.75  0.6666  0.625  0.7  0.5834  0.5714  0.5000  0.4444  0.40
Gaming         1.0  0.75  0.8334  0.750  0.6  0.5000  0.428

---
## 13. Simpan Artefak (Opsional)

Menyimpan semua embedding dan similarity matrix ke folder `output/` untuk penggunaan ulang.

In [18]:
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Simpan channel matrix dan nama channel
np.save(f'{OUTPUT_DIR}/channel_matrix.npy', channel_matrix)
pd.DataFrame({'nama_channel': channel_names}).to_csv(
    f'{OUTPUT_DIR}/channel_names.csv', index=False
)

# 2. Simpan similarity matrix
np.save(f'{OUTPUT_DIR}/similarity_matrix.npy', similarity_matrix)
similarity_df.to_csv(f'{OUTPUT_DIR}/similarity_matrix.csv')

# 3. Simpan hasil evaluasi
eval_df.to_csv(f'{OUTPUT_DIR}/evaluation_results.csv')

# 4. Simpan DataFrame hasil preprocessing
cols_to_save = ['id', 'nama_channel', 'kategori', 'judul',
                'judul_clean', 'judul_lower', 'judul_tokenized']
df[cols_to_save].to_csv(f'{OUTPUT_DIR}/preprocessed_data.csv', index=False)

print(f'Semua hasil disimpan ke folder: {OUTPUT_DIR}/')
for fname in os.listdir(OUTPUT_DIR):
    fsize = os.path.getsize(f'{OUTPUT_DIR}/{fname}') / 1024
    print(f'  {fname:<40} ({fsize:,.1f} KB)')
# ─── Tambahan untuk Flask App ────────────────────────────────
# Simpan video embeddings (dibutuhkan oleh web app)
np.save(f'{OUTPUT_DIR}/video_embeddings.npy', video_embeddings)

# Simpan metadata video lengkap (untuk ditampilkan di web app)
cols_meta = ['id', 'nama_channel', 'kategori', 'judul',
             'link', 'link_channel', 'jumlah_tayangan',
             'tanggal_upload', 'jumlah_pelanggan']
df[cols_meta].to_csv(f'{OUTPUT_DIR}/video_metadata.csv', index=False)

print(f'  video_embeddings.npy               ({os.path.getsize(f"{OUTPUT_DIR}/video_embeddings.npy")/1024:,.1f} KB)')
print(f'  video_metadata.csv                 ({os.path.getsize(f"{OUTPUT_DIR}/video_metadata.csv")/1024:,.1f} KB)')


Semua hasil disimpan ke folder: output/
  similarity_matrix.npy                    (39.2 KB)
  channel_names.csv                        (1.4 KB)
  evaluation_results.csv                   (1.5 KB)
  video_metadata.csv                       (2,047.0 KB)
  preprocessed_data.csv                    (2,778.0 KB)
  similarity_matrix.csv                    (103.7 KB)
  channel_matrix.npy                       (300.1 KB)
  video_embeddings.npy                     (30,000.1 KB)
  video_embeddings.npy               (30,000.1 KB)
  video_metadata.csv                 (2,047.0 KB)


---
## Ringkasan Metodologi

| No | Tahap | Metode | Catatan |
|---|---|---|---|
| 1 | Install & Import Library | Setup dependency dan import modul utama | `torch`, `transformers`, `stanza`, `sklearn`, `pandas`, `numpy`, `tqdm` |
| 2 | Load Dataset JSON | Membaca data dan membentuk DataFrame | `json`, `pandas` |
| 3 | Text Cleaning | Hapus emoji, simbol non-standar, encoding rusak | `re` |
| 4 | Case Folding | Lowercase seluruh teks | built-in |
| 5 | Tokenisasi dengan Stanza | Tokenisasi Bahasa Indonesia | `stanza` |
| 6 | Split Candidate & Query Sets | Stratified split 80/20 di level channel | `sklearn` |
| 7 | Load Model IndoBERT (Pre-trained) | Memuat tokenizer dan model `indolem/indobert-base-uncased` | `transformers`, `torch` (tanpa fine-tuning) |
| 8 | Embedding Judul Video | Masked Mean Pooling berbasis attention mask | `transformers`, `torch` |
| 9 | Agregasi Vektor Channel | Mean pooling antar video per channel + L2 normalization | `numpy`, `sklearn` |
| 10 | Similarity Matrix | Cosine similarity antar channel | `sklearn` |
| 11 | Rekomendasi | Top-K similarity tertinggi dari Candidate Set | `pandas`, `numpy` |
| 12 | Evaluasi | Precision@K pada Query Set (ground truth: kategori) | `numpy`, `pandas` |
| 13 | Simpan Artefak (Opsional) | Menyimpan matrix, metadata, dan hasil evaluasi | `numpy`, `pandas`, `os` |

**Pendekatan:** Content-Based Filtering (Unsupervised)  
**Model NLP:** `indolem/indobert-base-uncased` (Pre-trained, tanpa fine-tuning)  
**Fitur Utama:** Judul Video YouTube  

**Rangkuman Model:**
- Sistem ini **unsupervised**, sehingga split diposisikan sebagai **Candidate Set** dan **Query Set**, bukan train/test untuk training model.
- Fungsi rekomendasi dan evaluasi memakai parameter eksplisit (tanpa `global`) agar lebih clean dan reusable.
- **Batasan evaluasi:** relevansi berbasis kategori dapat mengabaikan kemiripan semantik lintas kategori yang sebenarnya dekat.